In [1]:
import sys
sys.path.append('../src')

In [2]:
print(sys.path)

['/home/minisemin/anaconda3/envs/pinn/lib/python311.zip', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11/lib-dynload', '', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11/site-packages', '../src']


In [3]:
# Import necessary libraries

import jax
import jax.numpy as jnp

# Configure JAX to use only the CPU
# jax.config.update('jax_platform_name', 'cpu') 

print('jax:', jax.__version__)      # 0.7.2
print('devices:', jax.devices())    # [CudaDevice(id=0)]

import jax.experimental.layout as _layout
if not hasattr(_layout, 'DeviceLocalLayout'):
        # Provide a minimal alias so orbax can import function annotations\n",
        _layout.DeviceLocalLayout = _layout.Layout
        print('Patched jax.experimental.layout.DeviceLocalLayout ->', getattr(_layout, 'DeviceLocalLayout'))
import os
import numpy as np
import matplotlib.pyplot as plt
from pinn.train import create_train_state, train_step
from pinn.cryoet_io import load_mrc_data
import pickle
from flax.training import checkpoints
from pinn.utils import initial_loss
from tqdm.notebook import trange

jax: 0.7.2
devices: [CudaDevice(id=0)]
Patched jax.experimental.layout.DeviceLocalLayout -> <class 'jax._src.layout.Layout'>


In [4]:
# Utility function to convert GPU/JAX arrays to CPU/NumPy arrays

def to_numpy(tree):
    return jax.tree_util.tree_map(
        lambda x: np.asarray(jax.device_get(x)) if isinstance(x, (jnp.ndarray, np.ndarray)) else x,
        tree,
    )

def as_f32_scalar(x):
    # Make sure to extract as a scalar (float32) even if it's an array/DeviceArray
    return np.asarray(x, dtype=np.float32).reshape(()).item()

In [ ]:
NUM_CHECKPOINTS_TO_KEEP = 1000 # Checkpoint retention count (older ones get removed)

# Set loss function weights
lambda_1 = 1000000  # Data loss weight
lambda_2 = 1000     # Physics loss weight
sdf_pretrain = "sphere"

# Initialize JAX random key
key = jax.random.PRNGKey(0)

# Create train state
state, model = create_train_state(key, lambda_1, lambda_2, sdf_pretrain, learning_rate=1e-3)
print(f"Using lambda_1: {state.lambda_1}, lambda_2: {state.lambda_2}")

In [ ]:
# Load MRC data
data_name = "74_downsampled"
data_path = f"../data/Chestnut_Jove_2024_downsampled/{data_name}.mrc"

data = load_mrc_data(data_path)
print(f"{data_name} loaded successfully! Shape:", data.shape)
GRID_X = data.shape[2]
GRID_Y = data.shape[1]
GRID_Z = data.shape[0]

In [ ]:
# Training data
num_collocation_points = 100000
x_train = jax.random.uniform(key, (num_collocation_points, 3), minval=-1.0, maxval=1.0)

# Training loop
num_steps = 10000
save_interval = 100

checkpoint_dir = os.path.abspath(f"../outputs/logs/{data_name}/lambda_{lambda_1}_{lambda_2}")